# Task 8 — Machine Learning: Predicting Final Grade (G3)

**Objective:** predict each student's final grade (G3, 0–20 scale) from background, school life and prior grades.

**Data:** Student Performance (UCI) — 649 students, Portuguese course.
**Models:** Linear Regression · Ridge · Random Forest · HistGradientBoosting, compared against naive baselines.
**Metrics:** RMSE · MAE · MedAE · R² · % within 1 point · failure recall / precision.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import (mean_squared_error, mean_absolute_error, median_absolute_error,
                             r2_score, precision_score, recall_score)
from sklearn.model_selection import (train_test_split, KFold, RepeatedKFold, RandomizedSearchCV,
                                     cross_val_score, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

N_TREES = 300
N_ITER_SEARCH = 10
N_ITER_NESTED = 8
N_REPEATS = 3

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def metrics(y_true, y_pred):
    """Global (RMSE, R2) and robust (MAE, MedAE, % within 1 point) metrics."""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    err = y_true - y_pred
    return {"RMSE": rmse(y_true, y_pred), "MAE": mean_absolute_error(y_true, y_pred),
            "MedAE": median_absolute_error(y_true, y_pred), "R²": r2_score(y_true, y_pred),
            "±1 pt (%)": 100 * float(np.mean(np.abs(err) <= 1))}

def predict_clipped(model, X_):
    return np.clip(model.predict(X_), 0, 20)
print("Setup complete.")

## 2. Data Loading

In [ ]:
DATA_FILE = Path("StudentPerformance.csv")

if not DATA_FILE.exists():
    print("StudentPerformance.csv not found. Downloading from UCI...")
    from ucimlrepo import fetch_ucirepo
    student = fetch_ucirepo(id=320)
    student.data.original.to_csv(DATA_FILE, index=False, encoding="utf-8-sig")

raw = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
print("Shape:", raw.shape)
assert raw.shape == (649, 33), (
    "Unexpected shape: this notebook expects the Portuguese file (649 x 33). "
    "The mathematics file has 395 rows.")
display(raw.head())

## 3. Data Quality Check

In [ ]:
raw.info()
print("\nMissing values:", int(raw.isnull().sum().sum()))
print("Duplicated rows: ", int(raw.duplicated().sum()))
display(raw[["G1", "G2", "G3"]].describe().T.round(2))

In [ ]:
dropouts = raw[raw["G3"] == 0]
print(f"G3 = 0: {len(dropouts)} students ({len(dropouts) / len(raw):.1%})")
print(f"G2 = 0: {(raw['G2'] == 0).sum()} students | G1 = 0: {(raw['G1'] == 0).sum()} students")
print("\nAmong students with G2 = 0, share with G3 = 0:",
      f"{(raw.loc[raw['G2'] == 0, 'G3'] == 0).mean():.0%}")
print("Among students with G3 = 0, share with G2 > 0:",
      f"{(dropouts['G2'] > 0).mean():.0%}")

print("\nGrade distribution around pass threshold (10):")
print(raw["G3"].value_counts().sort_index().loc[7:13].to_string())

> **Insight:** dropouts (G3 = 0) are a separate population — many had average prior grades. A few such cases dominate RMSE, so MAE and MedAE are also reported.

## 4. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(raw["G3"], bins=21, kde=False, ax=axes[0])
axes[0].set_title("G3 distribution (peak at 0: dropouts)")
axes[0].set_xlabel("G3 (0-20)")

plot_df = raw.assign(Status=np.where(raw["G3"] == 0, "Dropout (G3 = 0)", "Regular grade"))
sns.scatterplot(data=plot_df, x="G2", y="G3", hue="Status", alpha=0.6, ax=axes[1])
axes[1].plot([0, 19], [0, 19], "k--", lw=1, label="G3 = G2")
axes[1].set_title("G3 vs G2: strong linear relation + isolated dropouts")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=raw, x="studytime", y="G3", ax=axes[0])
axes[0].set_title("G3 by weekly study time")
axes[0].set_xlabel("studytime (1: <2h ... 4: >10h)")
sns.boxplot(data=raw, x="failures", y="G3", ax=axes[1])
axes[1].set_title("G3 by number of past failures")
plt.tight_layout()
plt.show()

> **Insight:** study time and past failures correlate with G3, but most of that signal is already captured by prior grades (G1/G2).

## 5. Features

Two features are added:
- `G_diff = G2 − G1`: grade trend (progression or decline);
- `G2_zero`: flag for already-visible disengagement in term 2.

In [ ]:
TARGET = "G3"

NUM_RAW = ["age", "Medu", "Fedu", "traveltime", "studytime", "failures", "famrel", "freetime",
           "goout", "Dalc", "Walc", "health", "absences", "G1", "G2"]
CAT = ["school", "sex", "address", "famsize", "Pstatus", "Mjob", "Fjob", "reason", "guardian",
       "schoolsup", "famsup", "paid", "activities", "nursery", "higher", "internet", "romantic"]
RAW_FEATURES = NUM_RAW + CAT

def make_features(df):
    """Feature engineering, used for training and new data."""
    d = df.copy()
    d["G_diff"] = d["G2"] - d["G1"]
    d["G2_zero"] = (d["G2"] == 0).astype("int64")
    return d

NUM = NUM_RAW + ["G_diff", "G2_zero"]
ALL_FEATURES = NUM + CAT

data = make_features(raw)
assert all(c in data.columns for c in ALL_FEATURES + [TARGET])
X, y = data[ALL_FEATURES], data[TARGET]
print(f"{len(NUM)} numeric + {len(CAT)} categorical = {len(ALL_FEATURES)} variables")

In [ ]:
num_corr = data[[TARGET] + NUM].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(num_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Correlation matrix (numeric variables)")
plt.show()
print("Correlation G3 ~ G2:", round(num_corr.loc["G3", "G2"], 3), "| G3 ~ G1:", round(num_corr.loc["G3", "G1"], 3),
      "| G1 ~ G2:", round(num_corr.loc["G1", "G2"], 3))

> **Insight:** G1 and G2 are highly correlated with each other and with G3. This is fine for prediction but splits importance between G1 and G2 (see Section 12).

## 6. Train / Test Split (stratified by grade)

80/20 split, stratified by G3 bands so dropouts and failing students are represented in both sets.

In [ ]:
bins = pd.cut(y, bins=[-1, 0, 9, 11, 13, 20], labels=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=bins)

print(f"Train: {len(X_train)} students | Test: {len(X_test)} students")
print(f"Dropouts (G3 = 0): train = {(y_train == 0).sum()} | test = {(y_test == 0).sum()}")
print(f"Failure rate (G3 < 10): train = {(y_train < 10).mean():.1%} | test = {(y_test < 10).mean():.1%}")

## 7. Pipelines and Naive Baselines

- **Baselines:** (a) predict the **mean**; (b) predict **G3 = G2** (prior term grade). ML is only useful if it beats (b).
- **Models:** Linear Regression, Ridge (internally cross-validated), Random Forest, HistGradientBoosting.
- Preprocessing is part of each pipeline (fitted on train only).

In [ ]:
def make_preprocessor(num_cols, cat_cols):
    numeric = Pipeline([("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler())])
    categorical = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", numeric, num_cols), ("cat", categorical, cat_cols)])


def build_pipelines(drop=()):
    """Build pipelines; `drop` removes features."""
    keep = lambda cols: [c for c in cols if c not in drop]
    pre = lambda: make_preprocessor(keep(NUM), keep(CAT))
    return {
        "Linear Regression": Pipeline([("preprocessor", pre()), ("model", LinearRegression())]),
        "Ridge": Pipeline([("preprocessor", pre()),
                           ("model", RidgeCV(alphas=np.logspace(-2, 3, 30)))]),
        "Random Forest": Pipeline([("preprocessor", pre()),
                                   ("model", RandomForestRegressor(n_estimators=N_TREES,
                                                                   random_state=RANDOM_STATE, n_jobs=1))]),
        "HistGradientBoosting": Pipeline([("preprocessor", pre()),
                                          ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),
    }

pipelines = build_pipelines()
dummy = Pipeline([("model", DummyRegressor(strategy="mean"))])

base_rows = {
    "Baseline: Mean": metrics(y_test, np.full(len(y_test), y_train.mean())),
    "Baseline: G3 = G2": metrics(y_test, X_test["G2"]),
}
display(pd.DataFrame(base_rows).T.round(3))

> **Insight:** `G3 = G2` is the true benchmark. G2 already explains most of G3, so ML gains are measured against it.

## 8. Repeated Cross-Validation and Paired Comparison

Repeated 5-fold CV on the training set. All models are evaluated on the exact same folds, so errors can be compared fold by fold.

In [ ]:
rkf = RepeatedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

def repeated_cv(model, X_, y_):
    """Per-fold RMSE and MAE (same folds for all models)."""
    r, m = [], []
    for tr, va in rkf.split(X_):
        fitted = clone(model).fit(X_.iloc[tr], y_.iloc[tr])
        p = predict_clipped(fitted, X_.iloc[va])
        r.append(rmse(y_.iloc[va], p)); m.append(mean_absolute_error(y_.iloc[va], p))
    return np.array(r), np.array(m)

cv_scores = {}
cv_scores["Baseline: G3 = G2"] = (
    np.array([rmse(y_train.iloc[va], X_train.iloc[va]["G2"]) for _, va in rkf.split(X_train)]),
    np.array([mean_absolute_error(y_train.iloc[va], X_train.iloc[va]["G2"]) for _, va in rkf.split(X_train)]))
cv_scores["Baseline: Mean"] = repeated_cv(dummy, X_train, y_train)
for name, pipe in pipelines.items():
    cv_scores[name] = repeated_cv(pipe, X_train, y_train)

ref = cv_scores["Baseline: G3 = G2"][0]
rows = []
for name, (r, m) in cv_scores.items():
    diff = r - ref
    rows.append({"Model": name, "RMSE CV": r.mean(), "± std": r.std(), "MAE CV": m.mean(),
                 "Δ RMSE vs G3=G2": diff.mean(), "% folds better than G3=G2": 100 * np.mean(diff < 0)})
cv_base = pd.DataFrame(rows).set_index("Model")
display(cv_base.round(3))

## 9. Hyperparameter Tuning (Random Forest and HistGradientBoosting)

In [ ]:
param_rf = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [None, 8, 15],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": [0.3, 0.5, 0.7, 1.0],
}
param_hgb = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_iter": [100, 200, 300],
    "model__max_leaf_nodes": [7, 15, 31],
    "model__min_samples_leaf": [10, 20, 40],
    "model__l2_regularization": [0.0, 0.1, 1.0],
}
GRIDS = {"Random Forest": param_rf, "HistGradientBoosting": param_hgb}

searches = {}
for name, grid in GRIDS.items():
    s = RandomizedSearchCV(pipelines[name], grid, n_iter=N_ITER_SEARCH,
                           cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
                           scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE,
                           n_jobs=-1, refit=True)
    s.fit(X_train, y_train)
    searches[name] = s
    print(f"{name:22s} | RMSE CV (optimistic, incl. selection) = {-s.best_score_:.3f}")
    print("   ", {k.replace('model__', ''): v for k, v in s.best_params_.items()})

## 9b. Nested Cross-Validation (unbiased estimate for tuned models)

The best CV score from tuning is optimistic (same data used to select and evaluate). Nested CV re-tunes inside each outer fold for an honest estimate.

In [ ]:
outer = KFold(5, shuffle=True, random_state=7)
inner = KFold(3, shuffle=True, random_state=RANDOM_STATE)

nested = {}
for name, grid in GRIDS.items():
    s = RandomizedSearchCV(pipelines[name], grid, n_iter=N_ITER_NESTED, cv=inner,
                           scoring="neg_root_mean_squared_error", random_state=RANDOM_STATE, n_jobs=-1)
    nested[name] = -cross_val_score(s, X_train, y_train, cv=outer, scoring="neg_root_mean_squared_error")
    print(f"{name:22s} | Nested RMSE = {nested[name].mean():.3f} ± {nested[name].std():.3f}")

## 10. Model Selection: One-Standard-Error Rule

When several models are statistically equivalent, the simplest is kept: easier to explain, more stable, cheaper. Any model within one standard error of the best is considered equivalent.

In [ ]:
honest = {
    "Linear Regression": cv_scores["Linear Regression"][0],
    "Ridge": cv_scores["Ridge"][0],
    "Random Forest": nested["Random Forest"],
    "HistGradientBoosting": nested["HistGradientBoosting"],
}
complexity = {"Linear Regression": 0, "Ridge": 1, "Random Forest": 2, "HistGradientBoosting": 3}

sel = pd.DataFrame({
    "RMSE (honest)": {k: v.mean() for k, v in honest.items()},
    "Std. error": {k: v.std() / np.sqrt(len(v)) for k, v in honest.items()},
    "Complexity": complexity,
})
best_row = sel["RMSE (honest)"].idxmin()
threshold = sel.loc[best_row, "RMSE (honest)"] + sel.loc[best_row, "Std. error"]
eligible = sel[sel["RMSE (honest)"] <= threshold]
best_name = eligible["Complexity"].idxmin()

display(sel.round(3))
print(f"Best raw RMSE: {best_row} | threshold (best + 1 std. error) = {threshold:.3f}")
print(f"Equivalent models: {list(eligible.index)}")
print(f"=> Selected model (simplest among equivalents): {best_name}")

## 11. Final Test Evaluation

Global metrics plus dropout-excluded metrics and business metrics: share of errors within 1 point and detection of failing students (G3 < 10).

In [ ]:
final_models = {name: pipe for name, pipe in pipelines.items()}
for name in GRIDS:
    final_models[name] = searches[name].best_estimator_
for name in ["Linear Regression", "Ridge"]:
    final_models[name].fit(X_train, y_train)

preds_test = {n: predict_clipped(m, X_test) for n, m in final_models.items()}
ok = (y_test > 0).values

rows = []
for name, p in preds_test.items():
    all_m = metrics(y_test, p)
    ok_m = metrics(y_test[ok], p[ok])
    fail_true, fail_pred = (y_test < 10).values, p < 9.5
    rows.append({
        "Model": name,
        "RMSE": all_m["RMSE"], "MAE": all_m["MAE"], "R²": all_m["R²"], "±1 pt (%)": all_m["±1 pt (%)"],
        "RMSE excl. dropouts": ok_m["RMSE"], "R2 excl. dropouts": ok_m["R²"],
        "Failure recall": recall_score(fail_true, fail_pred, zero_division=0),
        "Failure precision": precision_score(fail_true, fail_pred, zero_division=0),
    })
results = pd.DataFrame(rows).set_index("Model")
display(pd.concat([pd.DataFrame(base_rows).T[["RMSE", "MAE", "R²", "±1 pt (%)"]], results]).round(3))

In [ ]:
best_pred = preds_test[best_name]
sq_err = (y_test.values - best_pred) ** 2
share = sq_err[~ok].sum() / sq_err.sum()
print(f"Selected model: {best_name}")
print(f"Dropouts in test: {(~ok).sum()} of {len(ok)} ({(~ok).mean():.1%})")
print(f"-> they account for {share:.0%} of total squared error")
print(f"RMSE with dropouts: {rmse(y_test, best_pred):.3f} | excl. dropouts: {rmse(y_test[ok], best_pred[ok]):.3f}")

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
boot = {"RMSE": [], "MAE": [], "R²": []}
for _ in range(2000):
    idx = rng.integers(0, len(y_test), len(y_test))
    m = metrics(y_test.values[idx], best_pred[idx])
    for k in boot: boot[k].append(m[k])
ci = pd.DataFrame({k: np.percentile(v, [2.5, 50, 97.5]) for k, v in boot.items()},
                  index=["2.5%", "median", "97.5%"]).T
print(f"Bootstrap 95% confidence intervals — {best_name}")
display(ci.round(3))

## 12. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharex=True, sharey=True)
jit = np.random.default_rng(0).uniform(-0.15, 0.15, len(y_test))
for ax, name in zip(axes, ["Linear Regression", "Random Forest", "HistGradientBoosting"]):
    p = preds_test[name]
    ax.scatter(y_test + jit, p, alpha=0.5, s=18)
    ax.plot([0, 20], [0, 20], "r--", lw=1)
    ax.set_title(f"{name}\nR² = {r2_score(y_test, p):.3f}")
    ax.set_xlabel("Actual (with small jitter for readability)")
axes[0].set_ylabel("Predicted")
plt.tight_layout()
plt.show()

In [ ]:
resid = y_test.values - best_pred
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(resid, bins=25, ax=axes[0])
axes[0].axvline(0, color="r", ls="--")
axes[0].set_title(f"Residual distribution — {best_name}")
axes[0].set_xlabel("Residual (actual − predicted)")
axes[1].scatter(best_pred, resid, c=np.where(ok, "tab:blue", "tab:red"), alpha=0.6, s=20)
axes[1].axhline(0, color="r", ls="--")
axes[1].set_title("Residuals vs prediction (red: dropouts)")
axes[1].set_xlabel("Predicted G3")
plt.tight_layout()
plt.show()

top = X_test.assign(G3_actual=y_test, G3_pred=best_pred.round(1), Error=resid.round(1))
top = top.reindex(top["Error"].abs().sort_values(ascending=False).index).head(8)
print("Top 8 largest test errors:")
display(top[["G1", "G2", "G3_actual", "G3_pred", "Error", "absences", "failures", "studytime"]])

> **Insight:** excluding dropouts, residuals are narrow and centered. Extreme errors are almost always dropouts.

## 13. Training With vs Without Dropouts

Dropouts (G3 = 0) pull training downward for everyone. Comparison below is on non-dropout test students only.

In [ ]:
mask = (y_train > 0).values
rows = []
for name in ["Linear Regression", "Ridge", "Random Forest", "HistGradientBoosting"]:
    with_d = preds_test[name][ok]
    m2 = clone(final_models[name]).fit(X_train[mask], y_train[mask])
    without_d = predict_clipped(m2, X_test[ok])
    rows.append({"Model": name,
                 "RMSE (trained with dropouts)": rmse(y_test[ok], with_d),
                 "RMSE (trained without dropouts)": rmse(y_test[ok], without_d),
                 "MAE (with)": mean_absolute_error(y_test[ok], with_d),
                 "MAE (without)": mean_absolute_error(y_test[ok], without_d)})
display(pd.DataFrame(rows).set_index("Model").round(3))

> **Recommendation:** split the problem in two — (1) detect dropout risk (classification, or simple rule: `G2 = 0`), (2) predict grades for continuing students (regression trained without dropouts).

## 14. Feature Importance

Standard permutation underestimates G1 because G1 and G2 carry the same information. Grouped permutation {G1, G2} measures their joint contribution.

In [ ]:
perm = permutation_importance(final_models[best_name], X_test, y_test, n_repeats=15,
                              random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error", n_jobs=-1)
imp = (pd.DataFrame({"Feature": X_test.columns, "RMSE increase if permuted": perm.importances_mean,
                     "Std": perm.importances_std})
       .sort_values("RMSE increase if permuted", ascending=False))

grade_cols = ["G1", "G2", "G_diff", "G2_zero"]
base_rmse = rmse(y_test, predict_clipped(final_models[best_name], X_test))
gen = np.random.default_rng(RANDOM_STATE)
inc = []
for _ in range(15):
    Xp = X_test.copy()
    Xp[grade_cols] = X_test[grade_cols].values[gen.permutation(len(X_test))]
    inc.append(rmse(y_test, predict_clipped(final_models[best_name], Xp)) - base_rmse)
print(f"Grouped permutation of prior grades: +{np.mean(inc):.2f} of RMSE (±{np.std(inc):.2f})")

display(imp.head(12).round(3))
plt.figure(figsize=(8, 6))
t = imp.head(12).iloc[::-1]
plt.barh(t["Feature"], t["RMSE increase if permuted"], xerr=t["Std"])
plt.xlabel("RMSE increase (grade points)")
plt.title(f"Permutation importance — {best_name}")
plt.tight_layout()
plt.show()

> **Note:** variables with importance below their standard deviation are indistinguishable from noise. Low importance does not mean no causal effect — the effect may already be captured by G1/G2.

## 15. Three Prediction Times

- **Scenario A — after term 2:** G1 and G2 known (reliable final prediction);
- **Scenario B — after term 1:** only G1 known;
- **Scenario C — early warning:** no grades, context only.

Each scenario re-trains the same models without the unavailable variables.

In [ ]:
SCENARIOS = {
    "A — after P2 (G1, G2 known)": (),
    "B — after P1 (G1 only)": ("G2", "G_diff", "G2_zero"),
    "C — early warning (no grades)": ("G1", "G2", "G_diff", "G2_zero"),
}
rows = []
for sc_name, drop in SCENARIOS.items():
    mean_pred = np.full(len(y_test), y_train.mean())
    rows.append({"Scenario": sc_name, "Model": "Baseline: Mean",
                 **{k: v for k, v in metrics(y_test, mean_pred).items() if k in ("RMSE", "MAE", "R²")},
                 "Failure recall": 0.0})
    for name, pipe in build_pipelines(drop=drop).items():
        if name in searches:
            pipe.set_params(**{k: v for k, v in searches[name].best_params_.items() if k in pipe.get_params()})
        pipe.fit(X_train, y_train)
        p = predict_clipped(pipe, X_test)
        m = metrics(y_test, p)
        rows.append({"Scenario": sc_name, "Model": name, "RMSE": m["RMSE"], "MAE": m["MAE"], "R²": m["R²"],
                     "Failure recall": recall_score((y_test < 10).values, p < 9.5, zero_division=0)})
sc_df = pd.DataFrame(rows).set_index(["Scenario", "Model"])
display(sc_df.round(3))

> **Insight:** the earlier the prediction, the weaker the signal. Without grades, the model barely beats the mean — an early-warning tool should be used for prioritization, never as a reliable individual prediction.

## 16. Sensitivity to Sensitive Attributes

Sex, romantic relationship and parental education are sensitive sociodemographic attributes. We measure what they actually add.

In [ ]:
SENSITIVE = ("sex", "romantic", "Medu", "Fedu")
pipe_ns = build_pipelines(drop=SENSITIVE)[best_name]
if best_name in searches:
    pipe_ns.set_params(**{k: v for k, v in searches[best_name].best_params_.items() if k in pipe_ns.get_params()})
pipe_ns.fit(X_train, y_train)
p_ns = predict_clipped(pipe_ns, X_test)
cmp = pd.DataFrame({"With sensitive attributes": metrics(y_test, preds_test[best_name]),
                    "Without sensitive attributes": metrics(y_test, p_ns)}).T
display(cmp.round(3))

> **Note:** if the performance loss is negligible, remove these variables in production. In all cases, audit errors by subgroup (sex, school) before operational use.

## 17. Operational Predictions: Business Rule + Calibrated Interval

- **Business rule:** if `G2 = 0`, predict 0.
- **80% interval:** 10th/90th percentiles of out-of-sample train residuals, coverage verified on test.

In [ ]:
oof = np.clip(cross_val_predict(clone(final_models[best_name]), X_train, y_train,
                                cv=KFold(5, shuffle=True, random_state=RANDOM_STATE)), 0, 20)
q_low, q_high = np.quantile(y_train.values - oof, [0.10, 0.90])

lo_t = np.clip(best_pred + q_low, 0, 20); hi_t = np.clip(best_pred + q_high, 0, 20)
coverage = np.mean((y_test.values >= lo_t) & (y_test.values <= hi_t))
print(f"Out-of-sample train residual quantiles: 10 % = {q_low:.2f} | 90 % = {q_high:.2f}")
print(f"Empirical coverage on TEST: {coverage:.1%} (target: 80%)")

share_g2_zero = (raw.loc[raw["G2"] == 0, "G3"] == 0).mean() if (raw["G2"] == 0).any() else 0.0
RULE_G2_ZERO = share_g2_zero >= 0.9
print(f"Rule 'G2 = 0 -> G3 = 0' enabled : {RULE_G2_ZERO} (observed in {share_g2_zero:.0%} of cases)")


def predict_grade(model, new_raw, q=(q_low, q_high), rule=RULE_G2_ZERO):
    """Predict G3 from raw columns (same names as CSV)."""
    feats = make_features(new_raw)
    pred = np.clip(model.predict(feats[ALL_FEATURES]), 0, 20)
    low, high = np.clip(pred + q[0], 0, 20), np.clip(pred + q[1], 0, 20)
    if rule:
        zero = (feats["G2"] == 0).values
        pred, low, high = [np.where(zero, 0.0, a) for a in (pred, low, high)]
    out = new_raw[["school", "sex", "studytime", "failures", "absences", "G1", "G2"]].copy()
    out["Prediction"], out["Low (10%)"], out["High (90%)"] = pred.round(1), low.round(1), high.round(1)
    return out

In [ ]:
typical = dict(raw[RAW_FEATURES].mode().iloc[0])
new_students = pd.DataFrame([
    {**typical, "sex": "F", "studytime": 4, "failures": 0, "absences": 0, "G1": 16, "G2": 17},
    {**typical, "sex": "M", "studytime": 2, "failures": 0, "absences": 4, "G1": 11, "G2": 12},
    {**typical, "sex": "M", "studytime": 1, "failures": 2, "absences": 14, "G1": 7, "G2": 8},
    {**typical, "sex": "M", "studytime": 1, "failures": 3, "absences": 20, "G1": 5, "G2": 0},
])
profiles = ["Strong student", "Average student", "At-risk student", "Dropout (G2 = 0)"]

result = predict_grade(final_models[best_name], new_students)
result.insert(0, "Profile", profiles)
display(result)

## 18. Final Training and Saving

In [ ]:
final_pipeline = clone(final_models[best_name]).fit(X, y)

MODEL_FILE = "student_model_v2.pkl"
artifact = {
    "model": final_pipeline, "model_name": best_name,
    "features": ALL_FEATURES, "numeric": NUM, "categorical": CAT, "raw_features": RAW_FEATURES,
    "residual_quantiles_10_90": (float(q_low), float(q_high)),
    "target_range": (0, 20), "rule_g2_zero": bool(RULE_G2_ZERO),
    "input_format": "colonnes brutes du CSV 'portugais' (G1, G2 requises) ; make_features() ajoute G_diff et G2_zero",
}
joblib.dump(artifact, MODEL_FILE)

loaded = joblib.load(MODEL_FILE)
before = predict_grade(final_pipeline, new_students)["Prediction"].tolist()
after = predict_grade(loaded["model"], new_students, q=loaded["residual_quantiles_10_90"],
                      rule=loaded["rule_g2_zero"])["Prediction"].tolist()
assert before == after, "Reloaded model predictions differ!"
print(f"Model '{best_name}' saved to {MODEL_FILE} (identical predictions after reload)")

## 19. Conclusion

1. **G2 explains most of G3.** The `G3 = G2` baseline is already strong; ML adds only modest gains. Gaps smaller than the CV standard deviation are not significant.
2. **Linear ≈ Ridge ≈ Random Forest ≈ HGB:** on 649 students with an almost linear signal, complex models add little. The one-standard-error rule favors the simplest equivalent model.
3. **Dropouts dominate error.** A few G3 = 0 students drive most squared error. The fix is to separate dropout detection from grade prediction.
4. **Without prior grades, signal is weak (Scenario C):** early warning is for prioritization, not decision-making.
5. **Ethics:** drop sensitive attributes when performance allows and audit errors by subgroup.

**Limits:** single school system/country, single course (Portuguese), single year; small sample (~15 dropouts); integer grades with threshold effect at 10.

**Next steps:** dedicated dropout classifier; ordinal regression; merge with mathematics file; multi-year data; quantile models for heteroscedastic intervals.